In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [18]:
from datasets import load_dataset
from transformers import BertTokenizer , AutoModel, pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine


dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')['train']
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [4]:
# Q1: Load with HuggingFace datasets & create combined_text

def add_combined(row):
    row['combined_text'] = f"{row['prompt']} {row['A']}"
    return row

dataset = dataset.map(add_combined)

row_51 = dataset[51]
length = len(row_51['combined_text'])
print(f"combined_text at index 51: {row_51['combined_text']}")
print(f"Q1 - Character length: {length}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

combined_text at index 51: Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Q1 - Character length: 614


In [5]:
# Q2: BERT vocabulary size 
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size
print(f"\nQ2 - BERT vocab size: {vocab_size}")


Q2 - BERT vocab size: 30522


In [6]:
# Q3: [SEP] token ID 
sep_id = tokenizer.sep_token_id
print(f"\nQ3 - [SEP] token ID: {sep_id}")
print(f"     Verify: tokenizer.convert_ids_to_tokens({sep_id}) = {tokenizer.convert_ids_to_tokens(sep_id)}")


Q3 - [SEP] token ID: 102
     Verify: tokenizer.convert_ids_to_tokens(102) = [SEP]


In [7]:
# Q4: Tokenize entire prompt column 
all_prompts = list(dataset['prompt'])  

encoded = tokenizer(
    all_prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

shape = tuple(encoded['input_ids'].shape)
print(f"Q4 - input_ids tensor shape: {shape}")
print(f"     Interpretation: ({shape[0]} rows, {shape[1]} tokens per row)")

Q4 - input_ids tensor shape: (2000, 128)
     Interpretation: (2000 rows, 128 tokens per row)


In [8]:
# Q5: Attention head dimensionality 
hidden_size = 768
num_heads = 12
head_dim = hidden_size // num_heads
print(f"\nQ5 - Each attention head dimension: {head_dim}")
print(f"     ({hidden_size} hidden size / {num_heads} heads = {head_dim})")


Q5 - Each attention head dimension: 64
     (768 hidden size / 12 heads = 64)


In [9]:
# Q6: last_hidden_state shape for row ID 0

model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = dataset[0]['prompt']
inputs = tokenizer(row0_prompt, return_tensors='pt')
print(f"\n     Row 0 prompt: {row0_prompt}")
print(f"     Number of tokens (with [CLS] and [SEP]): {inputs['input_ids'].shape[1]}")

with torch.no_grad():
    outputs = model(**inputs)

lhs_shape = tuple(outputs.last_hidden_state.shape)
print(f"\nQ6 - last_hidden_state shape: {lhs_shape}")
print(f"     Interpretation: ({lhs_shape[0]} batch, {lhs_shape[1]} tokens, {lhs_shape[2]} hidden dims)")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



     Row 0 prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
     Number of tokens (with [CLS] and [SEP]): 31

Q6 - last_hidden_state shape: (1, 31, 768)
     Interpretation: (1 batch, 31 tokens, 768 hidden dims)


In [ ]:
# Q7: Sum of first 5 values

row0_prompt = dataset[0]['prompt']
inputs = tokenizer(row0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

cls_vector = outputs.last_hidden_state[0, 0, :]   # batch 0, token 0 ([CLS])
first5 = cls_vector[:5].tolist()
print(f"Q7 - First 5 [CLS] values: {first5}")
print(f"Q7 - Sum of first 5 values: {round(sum(first5), 4)}")

Q7 - First 5 [CLS] values: [-0.4676641821861267, -0.07544498145580292, -0.20190085470676422, -0.007064265664666891, -0.4480222165584564]
Q7 - Sum of first 5 values: -1.2001


In [11]:
# Q8: Attention weight

model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

sentence = "Light-ion fusion is a technique."
inputs2 = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])
print(f"\nQ8 - Tokens: {list(enumerate(tokens))}")

fusion_idx = tokens.index('fusion')
print(f"     'fusion' is at token index: {fusion_idx}")

with torch.no_grad():
    outputs2 = model_attn(**inputs2)

# attentions: tuple of (num_layers,) each shape (batch, heads, seq, seq)
last_layer_attn = outputs2.attentions[-1]        # last layer
head0_attn      = last_layer_attn[0, 0, :, :]   # head 0, all tokens
cls_to_fusion   = head0_attn[0, fusion_idx].item()
print(f"Q8 - Attention weight [CLS] → 'fusion': {round(cls_to_fusion, 4)}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Q8 - Tokens: [(0, '[CLS]'), (1, 'light'), (2, '-'), (3, 'ion'), (4, 'fusion'), (5, 'is'), (6, 'a'), (7, 'technique'), (8, '.'), (9, '[SEP]')]
     'fusion' is at token index: 4
Q8 - Attention weight [CLS] → 'fusion': 0.1025


In [12]:
# Q9: Sentence-Transformer cosine similarity
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row0 = dataset[0]
emb_prompt = st_model.encode(str(row0['prompt']), convert_to_tensor=True)
emb_B      = st_model.encode(str(row0['B']),      convert_to_tensor=True)
sim_score  = util.cos_sim(emb_prompt, emb_B).item()
print(f"\nQ9 - Cosine similarity (prompt vs B, row 0): {round(sim_score, 4)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Q9 - Cosine similarity (prompt vs B, row 0): 0.7658


In [13]:
# Q10: Building pipeline

def map_at_3(correct, preds):
    for k, p in enumerate(preds[:3]):
        if p == correct:
            return 1.0 / (k + 1)
    return 0.0

options = ['A', 'B', 'C', 'D', 'E']
answers = [str(r['answer']) for r in dataset]

# TF-IDF pipeline
print("Building TF-IDF pipeline...")
all_texts = [' '.join([str(row[c]) for c in ['prompt','A','B','C','D','E']]) for row in dataset]
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(all_texts)

tfidf_top3 = []
for row in dataset:
    pv = tfidf.transform([str(row['prompt'])])
    sims = {opt: sk_cosine(pv, tfidf.transform([str(row[opt])]))[0][0] for opt in options}
    tfidf_top3.append(sorted(sims, key=sims.get, reverse=True)[:3])

# MiniLM pipeline
print("Building MiniLM pipeline...")
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
prompts = [str(r['prompt']) for r in dataset]
prompt_embs = st_model.encode(prompts, batch_size=64, show_progress_bar=True, convert_to_tensor=True)

miniLM_top3 = []
for i, row in enumerate(dataset):
    opt_embs = st_model.encode([str(row[o]) for o in options], convert_to_tensor=True)
    sims = util.cos_sim(prompt_embs[i], opt_embs)[0].tolist()
    ranked = sorted(zip(options, sims), key=lambda x: x[1], reverse=True)
    miniLM_top3.append([o for o, _ in ranked[:3]])

# Results
map_miniLM = np.mean([map_at_3(a, p) for a, p in zip(answers, miniLM_top3)])
print(f"\nQ10 - MiniLM MAP@3: {round(map_miniLM, 4)}")

miniLM_wins = sum(
    1 for a, tf3, ml3 in zip(answers, tfidf_top3, miniLM_top3)
    if a not in tf3 and a in ml3
)
print(f"Q10 - Questions MiniLM recovers that TF-IDF misses: {miniLM_wins}")

Building TF-IDF pipeline...
Building MiniLM pipeline...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Q10 - MiniLM MAP@3: 0.4231
Q10 - Questions MiniLM recovers that TF-IDF misses: 502


In [ ]:
# Q11: Zero-shot classification
zsc = pipeline("zero-shot-classification")  

row1 = dataset[1]  # index 1 = 2nd row
sequence   = str(row1['prompt'])
candidates = [str(row1['A']), str(row1['B']), str(row1['C'])]

result_softmax = zsc(sequence, candidate_labels=candidates)
print("Q11 - Zero-shot (softmax) results:")
for label, score in zip(result_softmax['labels'], result_softmax['scores']):
    print(f"  {label[:60]}: {round(score, 4)}")
top_score = result_softmax['scores'][0]  # already sorted by score descending
print(f"\nQ11 - Top-ranked probability: {round(top_score, 4)}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11 - Zero-shot (softmax) results:
  Accelerator-based light-ion fusion is a technique that uses : 0.4575
  Accelerator-based light-ion fusion is a technique that uses : 0.2751
  Accelerator-based light-ion fusion is a technique that uses : 0.2675

Q11 - Top-ranked probability: 0.4575


In [ ]:
# Q12: Zero-shot classification
result_sigmoid = zsc(sequence, candidate_labels=candidates, multi_label=True)
print("\nQ12 - Zero-shot (sigmoid/multi_label) results:")
for label, score in zip(result_sigmoid['labels'], result_sigmoid['scores']):
    print(f"  {label[:60]}: {round(score, 4)}")

sum_softmax = sum(result_softmax['scores'])
sum_sigmoid = sum(result_sigmoid['scores'])
abs_diff    = abs(sum_softmax - sum_sigmoid)

print(f"\n  Sum of softmax probabilities : {round(sum_softmax, 4)}")
print(f"  Sum of sigmoid probabilities : {round(sum_sigmoid, 4)}")
print(f"Q12 - Absolute difference      : {round(abs_diff, 4)}")



Q12 - Zero-shot (sigmoid/multi_label) results:
  Accelerator-based light-ion fusion is a technique that uses : 0.0005
  Accelerator-based light-ion fusion is a technique that uses : 0.0
  Accelerator-based light-ion fusion is a technique that uses : 0.0

  Sum of softmax probabilities : 1.0
  Sum of sigmoid probabilities : 0.0005
Q12 - Absolute difference      : 0.9995


In [ ]:
# Q13: Generative QA 

tokenizer_flan = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_flan = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row0 = dataset[0]
input_str = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)
print(f"Q13 - Input string:\n{input_str}")

inputs = tokenizer_flan(input_str, return_tensors="pt")
outputs = model_flan.generate(**inputs, max_new_tokens=5)
result = tokenizer_flan.decode(outputs[0], skip_special_tokens=True)

print(f"\nQ13 - Model output: '{result}'")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 - Input string:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

Q13 - Model output: 'B'
